- ### regex

1. [Pattern](#pattern)

2. [Matching](#matching)

3. [Extract captured groups](#extract-captured-groups)

4. [Replace](#replace)


---

- ### Pattern

**Character classes**

```
.        any character
[abc]    a or b or c
[^abc]   NOT a or b or c
[a-z]    range
[0-9]    digit
```
**Predefined classes**

```
\d       digit      (same as [0-9])
\D       non-digit
\w       word char  (same as [A-Za-z0-9_])
\W       non-word
\s       whitespace
\S       non-whitespace
```
**Repetition**

```
a*       0 or more
a+       1 or more
a?       0 or 1
a{3}     exactly 3
a{2,4}   2 to 4
a{2,}    2 or more
```
**Anchors**

```
^        start of string
$        end of string
\b       word boundary
\B       not a word boundary
```
Groups
```
(a|b)    group with OR
(...)    capture group
(?:...)  non-capturing group
```


---

- ### Matching

In [2]:
# Setup for oneline command %%cpp
import os, tempfile, subprocess
from IPython.core.magic import register_cell_magic
import shlex

@register_cell_magic
def cpp(line, cell):
    """
    Usage:
    %%cpp -i "input for cin" -- arg1 arg2 ...
    """
    tokens = shlex.split(line)
    input_data = None
    run_args = []

    # Parse stdin input
    if "-i" in tokens:
        idx = tokens.index("-i")
        if idx + 1 < len(tokens):
            input_data = tokens[idx + 1]

    # Parse program arguments after --
    if "--" in tokens:
        idx = tokens.index("--")
        run_args = tokens[idx + 1:]

    # Write temp C++ file
    with tempfile.NamedTemporaryFile(suffix=".cpp", delete=False, mode="w") as tmp_cpp:
        tmp_cpp.write(cell)
        cpp_path = tmp_cpp.name
    exe_path = cpp_path[:-4] + ".exe"

    try:
        # Compile
        compile_proc = subprocess.run(
            ["g++", "-std=c++23", "-O2", "-Wall", cpp_path, "-o", exe_path],
            capture_output=True,
            text=True
        )
        if compile_proc.returncode != 0:
            print("❌ Compilation failed:\n", compile_proc.stderr)
            return

        # Run program
        run_proc = subprocess.run(
            [exe_path] + run_args,
            input=input_data,      # feed stdin here
            capture_output=True,
            text=True
        )
        if run_proc.stdout:
            print(run_proc.stdout, end="")
        if run_proc.stderr:
            print("⚠️ Runtime error:\n", run_proc.stderr)

    finally:
        for f in (cpp_path, exe_path):
            try: os.remove(f)
            except: pass

**matches the whole string**

In [10]:
%%cpp
#include <iostream>
#include <regex>
#include <string>
using namespace std;

int main() {
    regex re(R"(hello\s+hello\s+\d+\s+world)");
    if (regex_match("hello hello 123 world", re)) {
        cout << "Matched pattern!" << endl;
    } else {
        cout << "No match!" << endl;
    }
}

Matched pattern!


**matches substring**

In [ ]:
%%cpp
#include <iostream>
#include <regex>
#include <string>
using namespace std;

int main() {
    regex re(R"(hello)");
    if (regex_search("hello hello 123 world", re)) {
        cout << "Matched pattern!" << endl;
    } else {
        cout << "No match!" << endl;
    }
}

Matched pattern!


---

- ### Extract captured groups

**smatch**

In [ ]:
%%cpp
#include <iostream>
#include <regex>
#include <string>
using namespace std;

int main() {
    regex re(R"((\w+)@(\w+)\.(\w+))");
    smatch match;
    string email = "hello@world.com";
    if (regex_search(email, match, re)) {
        cout << "email: " << match[0] << endl; // entire match
        cout << "user: " << match[1] << endl;
        cout << "host: " << match[2] << endl;
    } else {
        cout << "No match!" << endl;
    }
}

email: hello@world.com
user: hello
host: world


**global search**

In [27]:
%%cpp
#include <iostream>
#include <regex>
#include <string>
using namespace std;

int main() {
    regex re(R"(\d+)");
    smatch m;
    string s = "hello 233 123 world 345";
    while (regex_search(s, m, re)) {
        cout << m[0] << endl; 
        s = m.suffix();
    }
}

233
123
345


---

- ### Replace

In [36]:
%%cpp
#include <iostream>
#include <regex>
#include <string>
#include <vector>
using namespace std;

int main() {
    regex re(R"((\d+)-(\d+)-(\d+))");
    vector<string> v = {"2000-01-02", "1999-12-31", "2024-06-15"};
    for (auto s : v) {
        s = regex_replace(s, re, "$2/$3/$1");
        cout << s << endl;
    }
}

01/02/2000
12/31/1999
06/15/2024
